# tmpl

> The template layer shared by claudedojo and codexdojo

In [ ]:
#| default_exp tmpl

#| export
The claudedojo and codexdojo launchers are mirror images: each captures a clean dojo round, gates it, stores it beside its metadata, and splices it into sessions at launch time. They differ only in the record shape their host uses (Claude Code transcript records vs Codex Responses items) and in transport. Everything representation-independent lives here instead of twice over there: the shared prompts and gate phrases, the completion-id receipt, the store layout, the launch-time loader and doc-state seeding, the content gates, and the round's structural boundaries. A backend contributes only its extractors, its transport, and its capture child.

In [ ]:
#| export
import json, os, re, subprocess, sys, tempfile
from importlib.resources import files
from fastcore.utils import *
from fastcore.script import call_parse
from clikernel.cli import _MARKER as _CLIK_MARKER
from aidialog.ipynb import read_ipynb, write_ipynb
from aidialog.hist import reply2dlg, dlg2reply, _parse_call
from aidialog.dialog import code_output, prompt_output
from llmdojo.rules import _state_root, _docnames

In [ ]:
from fastcore.test import *
import tempfile, shutil

## The shared script

Whichever host plays the round, the words around it are the same: the user prompt that opens a template, the prompt used when re-splicing after a compaction, the capture script a scripted child follows, and the script phrases that must never leak into a template's visible text.

In [ ]:
#| export
TMPL_PROMPT = "Bootstrap and complete the dojo and tell me when you're ready."
APPEND_PROMPT = "We've compacted - run the dojo round again and tell me when you're ready."
GATE_FORBID = ('recording a demo', 'these notes')
CAPTURE_SCRIPT = (files('llmdojo')/'dojo_data'/'capture_prompt.md').read_text()

## Completion receipts

A clean score prints a four-hex-character completion id, and that receipt is what launch-time registration honors. `find_cid` reads it from any iterable of record texts, so each backend passes its own text extractor's output. The dojo version rides along with every stored receipt: a version mismatch is how a stale template announces itself.

In [ ]:
#| export
def find_cid(
    txts, # Record texts, in order
):
    "The completion id receipt in a clean round's texts"
    return first(m[1] for t in txts if (m := re.search(r'Completion id: ([0-9a-f]{4})', t)))

def _dojo_v():
    from llmdojo.dojo import dojo_version
    return dojo_version()

In [ ]:
test_eq(find_cid(['strokes 8 = 8, par 8', 'Clean round. Completion id: ab12 - keep this id.']), 'ab12')
test_eq(find_cid(['no receipt here']), None)

## The template store

A store is one directory holding the round in the backend's native shape (`template.jsonl`) and the metadata needed to splice it (`meta.json`): the completion id, the dojo version it was played under, and the names `doc()`'d during the round. Items are stored as-is: the store neither knows nor cares whether a line is a Claude transcript record or a Codex Responses item.

In [ ]:
#| export
def save_store(
    d, # Store dir
    items, # Native template items or records
    cid, # The round's completion id receipt
    doced=None, # Names doc()'d during the baked round; the current conversation's doc-state if None
):
    "Write a template and its metadata to the store at `d`"
    if doced is None:
        from llmdojo.rules import doced as _cur
        doced = _cur()
    d = Path(d)
    d.mkdir(parents=True, exist_ok=True)
    (d/'template.jsonl').write_text(''.join(json.dumps(obj2dict(x))+'\n' for x in items))
    (d/'meta.json').write_text(json.dumps(dict(cid=cid, v=_dojo_v(), doced=list(doced))))

def load_store(
    d, # Store dir
):
    "The stored native items and metadata at `d`"
    d = Path(d)
    return (d/'template.jsonl').read_jsonl(), json.loads((d/'meta.json').read_text())

In [ ]:
items = [dict(role='user', content='hi'), dict(role='assistant', content='Completion id: ab12')]
store = Path(tempfile.mkdtemp())
save_store(store, items, 'ab12', doced=['rg','view_nb'])
back,meta = load_store(store)
test_eq(list(back), items)
test_eq(meta, dict(cid='ab12', v=_dojo_v(), doced=['rg','view_nb']))
shutil.rmtree(store)
meta

`load_reg` is the launch-time loader both backends wrap. The stores are package data - compiled beside the canonical dialog by `dojobuild` and shipped with the code - so there is nothing per-user to build or invalidate: updating llmdojo updates the template, and first launch on a fresh machine is a plain read. The loader warns on stderr when the packaged store's version doesn't match the installed tooling (a maintainer tripwire: the version was bumped without rerunning `dojobuild`; stdout is reserved - launchers print session ids for `$(...)` substitution), and registers the completion id so `dojo_start(cid)` honors the baked round. Registration writes machine-global state, so the backends' launch tests exercise it under a redirected state dir; there is no live example here.

In [ ]:
#| export
def load_reg(
    d, # Store dir; `default` if None
    default, # The backend's packaged store dir
    prog, # The backend's program name, for the skew warning
):
    "Load a template store, warn on version skew, and register its completion id"
    items,meta = load_store(Path(d or default))
    if meta['v'] != _dojo_v(): print(f"{prog}: template built under {meta['v']} but installed tooling is {_dojo_v()}; the baked round will not validate. Rebuild with dojobuild.", file=sys.stderr)
    from llmdojo.dojo import register_completion
    register_completion(meta['cid'], meta['v'])
    return items,meta

Every launch path also seeds doc-state, so the spliced round's claim to have read the docs is true on disk as well as in context. `merge=True` is for appending to a session that may already hold doc-state of its own (though after a compaction the hook has usually truncated it).

In [ ]:
#| export
def _seed_doced(sid, names, merge=False):
    "Make the baked round's doc-state true for conversation `sid`"
    d = _state_root()/'doced'
    d.mkdir(parents=True, exist_ok=True)
    f = d/f'{sid}.json'
    if merge and f.exists(): names = set(names) | set(json.loads(f.read_text()))
    f.write_text(json.dumps(sorted(names)))

## Content gates

A template must show exactly one round, dealt once and scored clean on the first try, with none of the capture script's phrasing leaking into the visible reply. These checks only need the round's kernel cell sources, its tool output texts, and the visible assistant text, so they are shared; representation-level checks (Claude's `is_error` results, Codex's orphaned call ids) stay with each backend's `is_clean`, which appends them to these.

In [ ]:
#| export
_START_RE = r'^dojo_start\(\s*\)\s*$'

def round_gates(
    cells, # The round's kernel cell sources, in order
    outs, # Its tool output texts
    visible, # Its visible assistant text, joined
    forbid=GATE_FORBID, # Phrases that must not appear in visible text
):
    "The content problems that disqualify a round from becoming a template, whichever host played it"
    probs = []
    if (n := sum(bool(re.match(_START_RE, c)) for c in cells)) != 1: probs.append(f'{n} dojo_start calls')
    if (n := sum(bool(re.match(r'^dojo_score\(', c)) for c in cells)) != 1: probs.append(f'{n} dojo_score calls (a clean round scores once, first try)')
    if any(re.match(r'^dojo_(?:redo|resume)\(', c) for c in cells): probs.append('needed a redo')
    if not any('Clean round' in o for o in outs): probs.append('no clean score')
    if (bad := [f for f in forbid if f in visible]): probs.append(f"script leaked into visible text: {', '.join(bad)}")
    return probs

A minimal clean round passes. Only a bare `dojo_start()` counts as dealing a round: `dojo_start('ab12')` replays a receipt and deals nothing, so it never affects the count.

In [ ]:
boot = ['doc(clik, pysk, edsk)', 'doc(dsk, exh, rgsk)', 'list_pyskills()']
round_cells = ['dojo_start()', '%cd /tmp/kata', '# kata 1', 'doc(report.daily_report)', "dojo_score(bash_calls=0, report='RB7034')"]
outs = ['...', 'Clean round. Completion id: ab12']
test_eq(round_gates(boot+round_cells, outs, 'OK ready.'), [])
test_eq(round_gates(["dojo_start('ab12')"]+boot+round_cells, outs, 'OK ready.'), [])

Each gate names its own failure: a redealt round, a rescore, a redo, a missing clean receipt, and script leakage are separate problems, reported together.

In [ ]:
test_eq(round_gates(['dojo_start()']+round_cells, outs, ''), ['2 dojo_start calls'])
test_eq(round_gates(round_cells+['dojo_score(1)'], outs, ''), ['2 dojo_score calls (a clean round scores once, first try)'])
test_eq(round_gates(round_cells+['dojo_redo(3)'], outs, ''), ['needed a redo'])
test_eq(round_gates(round_cells, ['...'], ''), ['no clean score'])
round_gates(round_cells, outs, 'as these notes say')

## The round's shape

A captured round has structural boundaries that need no sentinels or bookkeeping: the bootstrap `doc()` reads open it and the score closes it. `capture_slice` finds that span in a list of kernel cell sources - the last bootstrap before the last dealt round, through that round's first score - so an abandoned earlier round in the same conversation is skipped, and each backend only maps its native call/result pairs to cell sources before slicing.

In [ ]:
#| export
def capture_slice(
    cells, # Kernel cell sources of a conversation's calls, in order
):
    "The `(first, last)` cell indices of the captured round: bootstrap docs through the first score of the last dealt round"
    starts = [i for i,c in enumerate(cells) if re.match(_START_RE, c)]
    if not starts: raise ValueError('no dojo_start() call')
    start = starts[-1]
    boots = [i for i,c in enumerate(cells[:start]) if re.match(r'^doc\(clik,\s*pysk,\s*edsk\)\s*$', c)]
    if not boots: raise ValueError('no bootstrap doc call before dojo_start()')
    scores = [i for i,c in enumerate(cells[start:], start) if re.match(r'^dojo_score\(', c)]
    if not scores: raise ValueError('no dojo_score() after dojo_start()')
    return boots[-1], scores[0]

In [ ]:
test_eq(capture_slice(boot+round_cells), (0, 7))
test_eq(capture_slice(boot+round_cells+boot+round_cells), (8, 15))   # an abandoned round is skipped
test_fail(lambda: capture_slice(boot), contains='no dojo_start')
test_fail(lambda: capture_slice(round_cells), contains='no bootstrap doc call')
test_fail(lambda: capture_slice(boot+['dojo_start()']), contains='no dojo_score')

The names a template claims to have documented are readable from the round itself: every standalone `doc(...)` call in its cells, expanded to both spellings a call site might use. Deriving them from the cells means a template built from any source - a headless capture, a live session, a replay - carries the right doc-state with no side channel.

In [ ]:
#| export
def doced_names(
    cells, # Kernel cell sources
):
    "Names documented by standalone `doc(...)` calls in `cells`, in both spellings"
    docs = set()
    for c in cells:
        for m in re.finditer(r'(?m)^\s*doc\(([^()\n]+)\)\s*$', c):
            for name in m.group(1).split(','): docs.update(_docnames(name.strip()))
    return sorted(docs)

In [ ]:
doced_names(['doc(find_msgs, view_dlg)', 'x = 1', "doc(report.daily_report)\nreport.daily_report(SAMPLE)"])

## Refreshing a template

The round's cells rarely change, but their outputs go stale whenever the tooling's docs, card, or rendering move. Replaying the cells regenerates every receipt without model spend: a fresh `clikernel` CLI process (the same kernel, startup, magics, and rendering a live session gets) runs each cell in order in a scratch project, and the fresh outputs splice back into the template dialog. The kernel runs against the real machine state deliberately - the round deals into this machine's fixed run dir, and the replayed score's content-derived completion id is a registration we want. Stored templates never show that machine-specific path, though: on the way in, the canonical `/tmp/dojo` spelling is localized to the real run dir, and on the way out every occurrence maps back, so the committed artifact reads identically on every machine and refreshing an unchanged round on any box is diff-free (given the same installed tooling - environment-dependent outputs like the pyskill catalog still reflect the machine). Only the outputs move - cells and narration are inputs, so a change to the round itself is still capture-or-curation work.

In [ ]:
#| export
def _replay_cells(
    cells, # Kernel cell sources, run in order
    cwd, # Directory to start the kernel in
    env=None, # Extra environment entries for the kernel process
):
    "Each cell's rendered response from a fresh `clikernel` CLI at `cwd`"
    p = subprocess.Popen(['clikernel'], cwd=str(cwd), env=os.environ|(env or {}), text=True,
        stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)
    def upto(stop):
        res = []
        for line in p.stdout:
            if (line := line.rstrip('\n')) == stop: return res
            res.append(line)
        raise RuntimeError(f'clikernel exited during replay; last lines: {res[-5:]}')
    upto(_CLIK_MARKER)
    delim = p.stdout.readline().rstrip('\n')
    outs = []
    for c in cells:
        p.stdin.write(f'--\n{c}\n{delim}\n')
        p.stdin.flush()
        if (ack := p.stdout.readline().rstrip('\n')) != '.': raise RuntimeError(f'expected ack, got {ack!r}')
        outs.append('\n'.join(upto(delim)))
    p.stdin.write('exit\n')
    p.stdin.flush()
    p.wait()
    return outs

In [ ]:
rd = Path(tempfile.mkdtemp())
_replay_cells(['1+1', "print('hi')", 'x = 3'], rd)

`refresh_template` is the whole operation on a template dialog: parse each baked call back to its kernel cell, replay, splice each fresh output in (an empty response renders as the host's no-output line, matching what a live session records), re-derive `doced`, and gate with `round_gates` before writing. It refuses a dialog whose calls aren't all kernel cells, and a refresh that fails the gates writes nothing - if the round no longer plays clean, the cells need human attention, not fresher outputs.

In [ ]:
#| export
DOJO_CANON = '/tmp/dojo'

def canon_tmpl(
    dlg, # A template dialog whose reply holds a round
    canon=DOJO_CANON, # Canonical spelling for the round's run-dir path
):
    "Rewrite the round's run-dir path (read from its own `%cd` cell) to `canon` throughout the reply, so stored templates read the same on every machine"
    pm = dlg.messages[0]
    txt = pm.ai_res
    rd = first(re.findall(r"%cd ([^\s'\"\\]+)", txt))
    if rd and rd != canon: pm.output = prompt_output(txt.replace(rd, canon))
    return dlg

def refresh_template(
    src, # Path to a template dialog .ipynb
    dst=None, # Where to write the refreshed dialog; `src` if None
):
    "Replay `src`'s kernel cells through a fresh clikernel, splice in current outputs, refresh `doced`, and gate; writes and returns the refreshed (canonicalized) dialog"
    from llmdojo.dojo import _run_dir
    dlg = read_ipynb(str(src))
    pmsg = dlg.messages[0]
    sub = reply2dlg(pmsg)
    codes = [m for m in sub.messages if m.msg_type=='code']
    calls = [_parse_call(m.content) for m in codes]
    if bad := first(m.content for m,c in zip(codes,calls) if not (c and c[1].get('code'))): raise ValueError(f'not a kernel call: {bad}')
    cells = [c[1]['code'].replace(DOJO_CANON, str(_run_dir())) for c in calls]   # localize: the round plays in the real run dir
    with tempfile.TemporaryDirectory(prefix='dojorefresh_') as td:
        proj = Path(td)
        (proj/'pyproject.toml').write_text('[project]\nname = "dojo-refresh"\nversion = "0"\n')
        outs = _replay_cells(cells, proj)
    for m,(name,_),o in zip(codes, calls, outs): m.output = code_output(o or f'({name} completed with no output)')
    if probs := round_gates(cells, outs, ' '.join(m.content for m in sub.messages if m.msg_type=='note')): raise ValueError('; '.join(probs))
    pmsg.output = prompt_output(dlg2reply(sub).replace(str(_run_dir()), DOJO_CANON))   # canonicalize: the stored artifact reads the same everywhere
    dlg.meta['llmdojo'] = dict(doced=doced_names(cells))
    write_ipynb(dlg, str(dst or src))
    return dlg

The acceptance test replays the packaged round twice and demands byte-identical results - full-pipeline determinism with zero model spend. It runs against real machine state by design (see above): the completion id it registers is content-derived, so repeated runs re-register the same receipt.

In [ ]:
td = Path(tempfile.mkdtemp())
src = files('llmdojo')/'dojo_data'/'dojo_template.ipynb'
r1 = refresh_template(src, td/'r1.ipynb')
r2 = refresh_template(src, td/'r2.ipynb')
test_eq((td/'r1.ipynb').read_bytes(), (td/'r2.ipynb').read_bytes())
test_eq(r1.meta['llmdojo']['doced'], read_ipynb(str(src)).meta['llmdojo']['doced'])
assert DOJO_CANON in r1.messages[0].ai_res
from llmdojo.dojo import _run_dir
assert str(_run_dir()) not in r1.messages[0].ai_res
shutil.rmtree(td)
len(r1.messages)

## The dojobuild CLI

Template maintenance is one command, matching the layering: the launchers launch (and capture, which is backend-specific), while `dojobuild` owns the dialog-to-stores pipeline. Bare `dojobuild` is the common case - refresh the canonical dialog, then build both stores; chaining is safe because a refresh only regenerates machine-true receipts and `refresh_template` gates before writing. `--claude`/`--codex` build one store without refreshing: the post-review step after a capture or hand-curation.

In [ ]:
#| export
@call_parse(pos=['dialog'])
def main(
    dialog:str=None, # Template dialog path; the packaged dojo_data/dojo_template.ipynb where omitted
    claude:bool=False, # Build only the Claude store from the dialog, without refreshing
    codex:bool=False, # Build only the Codex store, without refreshing
):
    "Refresh the canonical template dialog (replaying its cells through a fresh clikernel), then build both stores"
    src = dialog or files('llmdojo')/'dojo_data'/'dojo_template.ipynb'
    from llmdojo import claudedojo, codexdojo
    if not (claude or codex):
        refresh_template(src)
        print(f'refreshed: {src}')
    if not codex:
        claudedojo.build_template(src)
        print(f'built: claude store ({claudedojo.TMPL_DIR})')
    if not claude:
        codexdojo.build_template(src)
        print(f'built: codex store ({codexdojo.TMPL_DIR})')

## Export -

In [ ]:
#|hide
#|eval: false
import nbdev; nbdev.nbdev_export()